# Normalizing Flows

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.get_logger().setLevel("ERROR")

# ============================================================
# 1) Robot parameters
# ============================================================
l1 = 1.0
l2 = 1.0

# ============================================================
# 2) Utility functions
# ============================================================
def wrap_to_pi(x):
    return (x + np.pi) % (2.0 * np.pi) - np.pi

def wrap_to_pi_tf(x):
    return tf.math.floormod(x + np.pi, 2.0 * np.pi) - np.pi

def forward_kinematics_np(q):
    theta1 = q[:, 0]
    theta2 = q[:, 1]

    px = l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2)
    py = l1 * np.sin(theta1) + l2 * np.sin(theta1 + theta2)

    return np.stack([px, py], axis=1)

def forward_kinematics_tf(q):
    theta1 = q[:, 0]
    theta2 = q[:, 1]

    px = l1 * tf.cos(theta1) + l2 * tf.cos(theta1 + theta2)
    py = l1 * tf.sin(theta1) + l2 * tf.sin(theta1 + theta2)

    return tf.stack([px, py], axis=1)

# ============================================================
# 3) Analytical IK: both branches
# ============================================================
def inverse_kinematics_two_branches(end_effector_values):
    px = end_effector_values[:, 0]
    py = end_effector_values[:, 1]

    r2 = px**2 + py**2
    c2 = (r2 - l1**2 - l2**2) / (2.0 * l1 * l2)
    c2 = np.clip(c2, -1.0, 1.0)

    s2_pos = np.sqrt(np.maximum(0.0, 1.0 - c2**2))
    s2_neg = -s2_pos

    theta2_up = np.arctan2(s2_pos, c2)
    theta2_down = np.arctan2(s2_neg, c2)

    def compute_theta1(theta2):
        k1 = l1 + l2 * np.cos(theta2)
        k2 = l2 * np.sin(theta2)
        return np.arctan2(py, px) - np.arctan2(k2, k1)

    theta1_up = compute_theta1(theta2_up)
    theta1_down = compute_theta1(theta2_down)

    q_up = np.stack([theta1_up, theta2_up], axis=1)
    q_down = np.stack([theta1_down, theta2_down], axis=1)

    q_up = wrap_to_pi(q_up)
    q_down = wrap_to_pi(q_down)

    return q_up.astype(np.float32), q_down.astype(np.float32)

# ============================================================
# 4) Dataset generation
# ============================================================
number_of_samples = 10000
np.random.seed(11)
tf.random.set_seed(11)

theta_values = np.random.uniform(-np.pi, np.pi, (number_of_samples, 2)).astype(np.float32)
p_values = forward_kinematics_np(theta_values).astype(np.float32)

q_up_all, q_down_all = inverse_kinematics_two_branches(p_values)

X_all = np.concatenate([p_values, p_values], axis=0).astype(np.float32)
Y_all = np.concatenate([q_up_all, q_down_all], axis=0).astype(np.float32)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_all, Y_all, test_size=0.2, random_state=42
)

X_eval = p_values.astype(np.float32)
q_up_eval, q_down_eval = inverse_kinematics_two_branches(X_eval)

# ============================================================
# 5) Normalize Cartesian input
# ============================================================
x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-6

X_train_n = ((X_train - x_mean) / x_std).astype(np.float32)
X_test_n = ((X_test - x_mean) / x_std).astype(np.float32)
X_eval_n = ((X_eval - x_mean) / x_std).astype(np.float32)

# ============================================================
# 6) Conditional coupling layer
# ============================================================
class ConditionalCoupling(tf.keras.layers.Layer):
    def __init__(self, mask, hidden_units=128):
        super().__init__()
        self.mask = tf.constant(mask, dtype=tf.float32)

        self.scale_net = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_units, activation="relu"),
            tf.keras.layers.Dense(hidden_units, activation="relu"),
            tf.keras.layers.Dense(2, kernel_initializer="zeros", bias_initializer="zeros")
        ])

        self.shift_net = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_units, activation="relu"),
            tf.keras.layers.Dense(hidden_units, activation="relu"),
            tf.keras.layers.Dense(2, kernel_initializer="zeros", bias_initializer="zeros")
        ])

    def forward(self, z, cond):
        z_masked = z * self.mask
        net_input = tf.concat([z_masked, cond], axis=1)

        log_s = self.scale_net(net_input) * (1.0 - self.mask)
        t = self.shift_net(net_input) * (1.0 - self.mask)

        # Stronger clamp for stability
        log_s = 0.8 * tf.tanh(log_s)

        x = z_masked + (1.0 - self.mask) * (z * tf.exp(log_s) + t)
        log_det = tf.reduce_sum(log_s, axis=1)

        return x, log_det

    def inverse(self, x, cond):
        x_masked = x * self.mask
        net_input = tf.concat([x_masked, cond], axis=1)

        log_s = self.scale_net(net_input) * (1.0 - self.mask)
        t = self.shift_net(net_input) * (1.0 - self.mask)

        log_s = 0.8 * tf.tanh(log_s)

        z = x_masked + (1.0 - self.mask) * ((x - t) * tf.exp(-log_s))
        log_det = -tf.reduce_sum(log_s, axis=1)

        return z, log_det

# ============================================================
# 7) Conditional RealNVP
# ============================================================
class ConditionalRealNVP(tf.keras.Model):
    def __init__(self, n_coupling_layers=6, hidden_units=128):
        super().__init__()
        self.coupling_layers = []

        for i in range(n_coupling_layers):
            if i % 2 == 0:
                mask = [1.0, 0.0]
            else:
                mask = [0.0, 1.0]
            self.coupling_layers.append(ConditionalCoupling(mask, hidden_units))

    def fwd(self, z, cond):
        log_det_total = tf.zeros(tf.shape(z)[0], dtype=tf.float32)
        x = z
        for layer in self.coupling_layers:
            x, log_det = layer.forward(x, cond)
            log_det_total += log_det
        return x, log_det_total

    def inv(self, x, cond):
        log_det_total = tf.zeros(tf.shape(x)[0], dtype=tf.float32)
        z = x
        for layer in reversed(self.coupling_layers):
            z, log_det = layer.inverse(z, cond)
            log_det_total += log_det
        return z, log_det_total

    def log_prob(self, x, cond):
        z, log_det = self.inv(x, cond)
        log_base = -0.5 * tf.reduce_sum(z**2 + tf.math.log(2.0 * np.pi), axis=1)
        return log_base + log_det

    def sample(self, cond, n_samples=100):
        batch_size = tf.shape(cond)[0]
        z = tf.random.normal((batch_size * n_samples, 2))
        cond_rep = tf.repeat(cond, repeats=n_samples, axis=0)

        x, _ = self.fwd(z, cond_rep)
        x = wrap_to_pi_tf(x)
        x = tf.reshape(x, (batch_size, n_samples, 2))
        return x

# ============================================================
# 8) Model and optimizer
# ============================================================
flow = ConditionalRealNVP(n_coupling_layers=6, hidden_units=128)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

dummy_x = tf.constant(X_train_n[:4], dtype=tf.float32)
dummy_y = tf.constant(Y_train[:4], dtype=tf.float32)
_ = flow.log_prob(dummy_y, dummy_x)

@tf.function
def train_step(cond_batch, q_batch):
    with tf.GradientTape() as tape:
        logp = flow.log_prob(q_batch, cond_batch)
        loss = -tf.reduce_mean(logp)

    if not tf.reduce_all(tf.math.is_finite(loss)):
        return tf.constant(np.nan, dtype=tf.float32)

    grads = tape.gradient(loss, flow.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 5.0)
    optimizer.apply_gradients(zip(grads, flow.trainable_variables))
    return loss

@tf.function
def val_step(cond_batch, q_batch):
    logp = flow.log_prob(q_batch, cond_batch)
    loss = -tf.reduce_mean(logp)
    return loss

# ============================================================
# 9) Data pipeline
# ============================================================
batch_size = 256
epochs = 80

train_size = len(X_train_n)
val_size = len(X_test_n)

steps_per_epoch = int(np.ceil(train_size / batch_size))
validation_steps = int(np.ceil(val_size / batch_size))

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_n, Y_train))
    .shuffle(min(10000, train_size), reshuffle_each_iteration=True)
    .batch(batch_size, drop_remainder=False)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_test_n, Y_test))
    .batch(batch_size, drop_remainder=False)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)

train_iter = iter(train_ds)
val_iter = iter(val_ds)

In [3]:
best_val = np.inf
best_weights = None

for epoch in range(epochs):
    train_losses = []
    skipped_batches = 0

    for _ in range(steps_per_epoch):
        cond_batch, q_batch = next(train_iter)
        loss = train_step(cond_batch, q_batch)

        loss_val = float(loss.numpy())
        if np.isfinite(loss_val):
            train_losses.append(loss_val)
        else:
            skipped_batches += 1

    val_losses = []
    for _ in range(validation_steps):
        cond_batch, q_batch = next(val_iter)
        loss = val_step(cond_batch, q_batch)
        loss_val = float(loss.numpy())
        if np.isfinite(loss_val):
            val_losses.append(loss_val)

    train_loss_mean = np.mean(train_losses) if len(train_losses) > 0 else np.nan
    val_loss_mean = np.mean(val_losses) if len(val_losses) > 0 else np.nan

    if np.isfinite(val_loss_mean) and val_loss_mean < best_val:
        best_val = val_loss_mean
        best_weights = flow.get_weights()

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train NLL: {train_loss_mean:.6f} | "
        f"Val NLL: {val_loss_mean:.6f} | "
        f"Skipped: {skipped_batches}"
    )

if best_weights is not None:
    flow.set_weights(best_weights)

# ============================================================
# 11) Prediction helpers
# ============================================================
def pick_best_sample_by_fk(cond_xy_normalized, cond_xy_original, samples):
    N = tf.shape(samples)[0]
    K = tf.shape(samples)[1]

    samples_flat = tf.reshape(samples, (-1, 2))
    fk_flat = forward_kinematics_tf(samples_flat)
    fk_all = tf.reshape(fk_flat, (N, K, 2))

    cond_expanded = tf.expand_dims(cond_xy_original, axis=1)
    fk_err = tf.norm(fk_all - cond_expanded, axis=2)

    best_idx = tf.argmin(fk_err, axis=1, output_type=tf.int32)
    batch_idx = tf.range(N, dtype=tf.int32)
    gather_idx = tf.stack([batch_idx, best_idx], axis=1)

    best_q = tf.gather_nd(samples, gather_idx)
    return best_q

def predict_one_solution(px_py, n_samples=200):
    px_py = px_py.reshape(1, 2).astype(np.float32)
    cond_n = ((px_py - x_mean) / x_std).astype(np.float32)

    cond_n_tf = tf.convert_to_tensor(cond_n, dtype=tf.float32)
    cond_tf = tf.convert_to_tensor(px_py, dtype=tf.float32)

    samples = flow.sample(cond_n_tf, n_samples=n_samples)
    q_best = pick_best_sample_by_fk(cond_n_tf, cond_tf, samples)
    return q_best.numpy()[0]

Epoch 001 | Train NLL: 4.701960 | Val NLL: 4.211883 | Skipped: 0
Epoch 002 | Train NLL: 3.934364 | Val NLL: 3.708271 | Skipped: 0
Epoch 003 | Train NLL: 3.536459 | Val NLL: 3.361398 | Skipped: 0
Epoch 004 | Train NLL: 3.190217 | Val NLL: 3.014787 | Skipped: 0
Epoch 005 | Train NLL: 2.822209 | Val NLL: 2.623033 | Skipped: 0
Epoch 006 | Train NLL: 2.379279 | Val NLL: 2.130781 | Skipped: 0
Epoch 007 | Train NLL: 1.869940 | Val NLL: 1.674754 | Skipped: 0
Epoch 008 | Train NLL: 1.509912 | Val NLL: 1.388997 | Skipped: 0
Epoch 009 | Train NLL: 1.215497 | Val NLL: 1.090502 | Skipped: 0
Epoch 010 | Train NLL: 0.977039 | Val NLL: 0.924814 | Skipped: 0
Epoch 011 | Train NLL: 0.875208 | Val NLL: 0.851219 | Skipped: 0
Epoch 012 | Train NLL: 0.808244 | Val NLL: 0.798847 | Skipped: 0
Epoch 013 | Train NLL: 0.743574 | Val NLL: 0.736958 | Skipped: 0
Epoch 014 | Train NLL: 0.677877 | Val NLL: 0.662264 | Skipped: 0
Epoch 015 | Train NLL: 0.621961 | Val NLL: 0.614280 | Skipped: 0
Epoch 016 | Train NLL: 0.

In [4]:
def branch_invariant_joint_error(q_pred, q_up, q_down):
    err_up = np.linalg.norm(wrap_to_pi(q_pred - q_up), axis=1)
    err_down = np.linalg.norm(wrap_to_pi(q_pred - q_down), axis=1)
    return np.minimum(err_up, err_down)

X_eval_tf_n = tf.convert_to_tensor(X_eval_n, dtype=tf.float32)
X_eval_tf = tf.convert_to_tensor(X_eval, dtype=tf.float32)

samples_eval = flow.sample(X_eval_tf_n, n_samples=100)
q_pred_eval = pick_best_sample_by_fk(X_eval_tf_n, X_eval_tf, samples_eval).numpy()

joint_err = branch_invariant_joint_error(q_pred_eval, q_up_eval, q_down_eval)
fk_pred_eval = forward_kinematics_np(q_pred_eval)
cart_err = np.linalg.norm(fk_pred_eval - X_eval, axis=1)

print("\nEvaluation results")
print("Mean branch-invariant joint error :", np.mean(joint_err))
print("Median branch-invariant joint error:", np.median(joint_err))
print("Mean Cartesian FK error           :", np.mean(cart_err))
print("Success rate (< 0.05 rad)         :", np.mean(joint_err < 0.05))


Evaluation results
Mean branch-invariant joint error : 0.05748272
Median branch-invariant joint error: 0.026862713
Mean Cartesian FK error           : 0.021046225
Success rate (< 0.05 rad)         : 0.7303


In [5]:
test_point = np.array([1, 1], dtype=np.float32)
q_hat = predict_one_solution(test_point, n_samples=300)

q_up_true, q_down_true = inverse_kinematics_two_branches(test_point.reshape(1, 2))

print("\nExample")
print("Target point      :", test_point)
print("Predicted q       :", q_hat)
print("True branch up    :", q_up_true[0])
print("True branch down  :", q_down_true[0])
print("Metric for sample :", branch_invariant_joint_error(
    q_hat.reshape(1, 2), q_up_true, q_down_true
)[0])


Example
Target point      : [1. 1.]
Predicted q       : [0.00262213 1.5741699 ]
True branch up    : [0.        1.5707963]
True branch down  : [ 1.5707963 -1.5707964]
Metric for sample : 0.0042728074
